## Exploration Findings

### Population

- `train_v2` contains 970,960 labelled users.
- `members_v3` contains approximately 6.77 million member records.
- `transactions_v2` contains 1,431,009 transaction records.
- `user_logs_v2` contains 18,396,362 listening records.

### Churn

- 883,630 users did not churn.
- 87,330 users churned.
- Churn rate is approximately 9%.

### Date ranges

- Transactions: 2015-01-01 to 2017-03-31.
- Listening activity: 2017-03-01 to 2017-03-31.

The raw dates are stored as `YYYYMMDD` integers. The unusual decimal boundaries seen in the data profiler are a result of treating these integers as continuous numeric values, not malformed dates.

### Missing values

Missing values occur only in `members_v3.gender`.

- `gender`: 4,429,505 missing values (65.43%).
- All other fields across the four source tables have no missing values.

### Duplicates and grain

The observed/expected grain is:

| Source | Grain |
|---|---|
| `train_v2` | One row per labelled user |
| `members_v3` | One row per member |
| `transactions_v2` | One row per transaction |
| `user_logs_v2` | One row per user per day |

`train_v2` was verified to contain one unique row per `msno`.

`transactions_v2` contains 1,431,009 rows but only 1,197,050 unique users, confirming that users can have multiple transaction records. This is expected and should not be treated as duplication.

A full duplicate-key scan for `members_v3` and `user_logs_v2` was not completed because the initial memory-safe approaches were too slow for the available hardware. Therefore, no claim of zero duplicates is made for those sources at this stage.

### Invalid values

- `is_churn` contains only `0` and `1`.
- `is_auto_renew` contains only `0` and `1`.
- `is_cancel` contains only `0` and `1`.
- No negative `payment_plan_days` values were found.
- No negative prices were found.
- No negative listening counts or listening durations were found.
- Transaction and listening dates all match the expected 8-digit `YYYYMMDD` format.
- `members_v3.bd` contains suspicious values: 274 negative values and 4,540,215 zero values, with a range of -7168 to 2016. This field requires investigation before being used in analysis.

### Transformation implications

The exploration indicates that:

1. `gender` has substantial missingness and should not be used without evaluating its analytical value and treatment of missing values.
2. `bd` requires further investigation before inclusion.
3. Transaction records must be aggregated to user level before being joined to the churn labels.
4. Listening records must be aggregated from user-day level to user level for the main churn analysis.
5. Date fields should be converted from `YYYYMMDD` integers to proper dates during the transformation stage.
6. Raw source files should remain unchanged.

In [2]:
from pathlib import Path
import pandas as pd

RAW_DIR = Path("../data/raw")

TRAIN = RAW_DIR / "train_v2.csv"
MEMBERS = RAW_DIR / "members_v3.csv"
TRANSACTIONS = RAW_DIR / "transactions_v2.csv"
USER_LOGS = RAW_DIR / "user_logs_v2.csv"

In [3]:
CHUNK_SIZE = 100_000

In [4]:
for path in [TRAIN, MEMBERS, TRANSACTIONS, USER_LOGS]:
    if not path.exists():
        raise FileNotFoundError(f"{path} does not exist. Please check the path.")
    else:
        print(f"{path.name}: {path.stat().st_size / (1024**2):.1f} MB")

train_v2.csv: 43.5 MB
members_v3.csv: 408.1 MB
transactions_v2.csv: 110.0 MB
user_logs_v2.csv: 1365.2 MB


In [5]:
for path in [TRAIN, MEMBERS, TRANSACTIONS, USER_LOGS]:
    header = pd.read_csv(path, nrows=0)
    print(f"\n{path.name}")
    print(header.columns.tolist())


train_v2.csv
['msno', 'is_churn']

members_v3.csv
['msno', 'city', 'bd', 'gender', 'registered_via', 'registration_init_time']

transactions_v2.csv
['msno', 'payment_method_id', 'payment_plan_days', 'plan_list_price', 'actual_amount_paid', 'is_auto_renew', 'transaction_date', 'membership_expire_date', 'is_cancel']

user_logs_v2.csv
['msno', 'date', 'num_25', 'num_50', 'num_75', 'num_985', 'num_100', 'num_unq', 'total_secs']


How many users?

In [6]:
train = pd.read_csv(TRAIN)
print("Users:", train['msno'].nunique())

Users: 970960


Churn Distribution

In [7]:
print(train["is_churn"].value_counts())
print(train["is_churn"].value_counts(normalize=True))

is_churn
0    883630
1     87330
Name: count, dtype: int64
is_churn
0    0.910058
1    0.089942
Name: proportion, dtype: float64


How many transactions?

In [9]:
transaction_rows = 0
for chunk in pd.read_csv(TRANSACTIONS, chunksize=CHUNK_SIZE):
    transaction_rows+=len(chunk)
print(f"Total rows in transactions: {transaction_rows}")

Total rows in transactions: 1431009


How many listening records?

In [11]:
log_rows=0

for chunk in pd.read_csv(USER_LOGS, chunksize=CHUNK_SIZE):
    log_rows+=len(chunk)
print(f"Total rows in user logs: {log_rows}")

Total rows in user logs: 18396362


Date Ranges

In [14]:
# For transactions:
min_transaction_date = None
max_transaction_date = None

for chunk in pd.read_csv(TRANSACTIONS, usecols=["transaction_date"], chunksize=CHUNK_SIZE, ):
    current_min = chunk["transaction_date"].min()
    current_max = chunk["transaction_date"].max()
    if min_transaction_date is None or current_min < min_transaction_date:
        min_transaction_date = current_min
    if max_transaction_date is None or current_max > max_transaction_date:
        max_transaction_date = current_max
print(min_transaction_date, max_transaction_date)

20150101 20170331


In [15]:
# For listening
min_log_date = None
max_log_date = None

for chunk in pd.read_csv(USER_LOGS, usecols=["date"], chunksize=CHUNK_SIZE):
    current_min = chunk["date"].min()
    current_max = chunk["date"].max()
    if min_log_date is None or current_min < min_log_date:
        min_log_date = current_min
    if max_log_date is None or current_max > max_log_date:
        max_log_date = current_max
print(min_log_date, max_log_date)

20170301 20170331


Missing Values

In [16]:
def profile_missing(path, chunksize=CHUNK_SIZE):
    missing = {}
    total_rows = 0

    for chunk in pd.read_csv(path, chunksize=chunksize):
        total_rows += len(chunk)
        for col in chunk.columns:
            missing[col] = missing.get(col, 0) + chunk[col].isna().sum()

    result = pd.DataFrame({
        "missing_count": missing.values(),
    }, index = missing.keys())

    result["missing_percentage"] = result["missing_count"] / total_rows * 100

    return total_rows, result.sort_values(
        "missing_count",
        ascending=False
    )

In [17]:
for name, path in {
    "members_v3": MEMBERS,
    "transactions_v2": TRANSACTIONS,
    "user_logs_v2": USER_LOGS
}.items():

    rows, missing = profile_missing(path)

    print(f"\n{name}")
    print(f"Rows: {rows:,}")
    display(missing)


members_v3
Rows: 6,769,473


,missing_count,missing_percentage
gender,4429505,65.433528
msno,0,0.000000
city,0,0.000000
bd,0,0.000000
registered_via,0,0.000000
registration_init_time,0,0.000000



transactions_v2
Rows: 1,431,009


,missing_count,missing_percentage
msno,0,0.0
payment_method_id,0,0.0
payment_plan_days,0,0.0
plan_list_price,0,0.0
actual_amount_paid,0,0.0
is_auto_renew,0,0.0
transaction_date,0,0.0
membership_expire_date,0,0.0
is_cancel,0,0.0



user_logs_v2
Rows: 18,396,362


,missing_count,missing_percentage
msno,0,0.0
date,0,0.0
num_25,0,0.0
num_50,0,0.0
num_75,0,0.0
num_985,0,0.0
num_100,0,0.0
num_unq,0,0.0
total_secs,0,0.0


In [18]:
print("train_v2")
display(
    train.isna()
    .sum()
    .to_frame("missing_count")
    .assign(
        missing_pct=lambda x: x["missing_count"] / len(train) * 100
    )
)

train_v2


,missing_count,missing_pct
msno,0,0.0
is_churn,0,0.0


### Missing-value findings

`members_v3` contains missing values only in `gender`.

- `gender`: 4,429,505 missing values (65.43%)
- `msno`, `city`, `bd`, `registered_via`, and `registration_init_time`: no missing values.

The high missingness in `gender` will be considered when deciding whether to include the field in the analytical model.

Missingness is evaluated separately from invalid values; fields such as `bd` may contain invalid values despite having no missing entries.

Duplicates

In [19]:
print("train_v2")
print("Rows:", len(train))
print("Unique msno:", train["msno"].nunique())
print("Duplicate msno:", train["msno"].duplicated().sum())

train_v2
Rows: 970960
Unique msno: 970960
Duplicate msno: 0


In [33]:
import importlib.util
print("DuckDB installed:", importlib.util.find_spec("duckdb") is not None)

DuckDB installed: False


In [ ]:
# Caution don't run this on the entire members_v3.csv at once, it may consume too much memory. Instead, we will read it in chunks and keep track of unique IDs and duplicates.

member_ids = set()
member_rows = 0
member_duplicate_ids = 0

for chunk in pd.read_csv(
    MEMBERS,
    usecols=["msno"],
    chunksize=CHUNK_SIZE           
):
    member_rows+=len(chunk)
    ids = chunk["msno"]

    # duplicated within this chunk
    member_duplicate_ids+=ids.duplicated().sum()

    # IDs already encountered in previous chunks
    member_duplicate_ids+=ids[ids.isin(member_ids)].nunique()

    member_ids.update(ids)

print("Rows:", member_rows)
print("Unique msno:", len(member_ids))
print("Duplicate msno:", member_duplicate_ids)

KeyboardInterrupt: 

In [22]:
transaction_users = set()
transaction_rows = 0

for chunk in pd.read_csv(
    TRANSACTIONS,
    usecols=["msno"],
    chunksize=CHUNK_SIZE
):
    transaction_rows += len(chunk)
    transaction_users.update(chunk["msno"])

print("Rows:", transaction_rows)
print("Unique users:", len(transaction_users))
print("Rows beyond unique users:", transaction_rows - len(transaction_users))

Rows: 1431009
Unique users: 1197050
Rows beyond unique users: 233959


In [23]:
import sqlite3

conn = sqlite3.connect("../data/exploration_temp.db")

conn.execute("""
    CREATE TABLE IF NOT EXISTS user_log_keys (
        msno TEXT,
        date INTEGER,
        PRIMARY KEY (msno, date)
    )
""")

conn.commit()

In [24]:
duplicate_log_rows = 0
log_rows = 0

for chunk in pd.read_csv(
    USER_LOGS,
    usecols=["msno", "date"],
    chunksize=CHUNK_SIZE
):
    log_rows += len(chunk)

    for row in chunk.itertuples(index=False):
        try:
            conn.execute(
                "INSERT INTO user_log_keys (msno, date) VALUES (?, ?)",
                (row.msno, int(row.date))
            )
        except sqlite3.IntegrityError:
            duplicate_log_rows += 1

    conn.commit()

print("Listening rows:", log_rows)
print("Duplicate msno + date rows:", duplicate_log_rows)

KeyboardInterrupt: 

In [25]:
conn.close()

Invalid Values

In [26]:
print("Invalid churn values:")
print(train.loc[~train["is_churn"].isin([0, 1]), "is_churn"].value_counts())

Invalid churn values:
Series([], Name: count, dtype: int64)


In [27]:
print("Unique churn values:", train["is_churn"].unique())

Unique churn values: [1 0]


In [28]:
bd_min = float("inf")
bd_max = float("-inf")
bd_negative = 0
bd_zero = 0

for chunk in pd.read_csv(
    MEMBERS,
    usecols=["bd"],
    chunksize=CHUNK_SIZE
):
    bd_min = min(bd_min, chunk["bd"].min())
    bd_max = max(bd_max, chunk["bd"].max())
    bd_negative += (chunk["bd"] < 0).sum()
    bd_zero += (chunk["bd"] == 0).sum()

print("bd min:", bd_min)
print("bd max:", bd_max)
print("bd negative:", bd_negative)
print("bd zero:", bd_zero)

bd min: -7168
bd max: 2016
bd negative: 274
bd zero: 4540215


In [29]:
invalid_auto_renew = 0
invalid_cancel = 0
negative_plan_days = 0
negative_list_price = 0
negative_actual_paid = 0

for chunk in pd.read_csv(
    TRANSACTIONS,
    usecols=[
        "payment_plan_days",
        "plan_list_price",
        "actual_amount_paid",
        "is_auto_renew",
        "is_cancel"
    ],
    chunksize=CHUNK_SIZE
):
    invalid_auto_renew += (~chunk["is_auto_renew"].isin([0, 1])).sum()
    invalid_cancel += (~chunk["is_cancel"].isin([0, 1])).sum()

    negative_plan_days += (chunk["payment_plan_days"] < 0).sum()
    negative_list_price += (chunk["plan_list_price"] < 0).sum()
    negative_actual_paid += (chunk["actual_amount_paid"] < 0).sum()

print("Invalid is_auto_renew:", invalid_auto_renew)
print("Invalid is_cancel:", invalid_cancel)
print("Negative payment_plan_days:", negative_plan_days)
print("Negative plan_list_price:", negative_list_price)
print("Negative actual_amount_paid:", negative_actual_paid)

Invalid is_auto_renew: 0
Invalid is_cancel: 0
Negative payment_plan_days: 0
Negative plan_list_price: 0
Negative actual_amount_paid: 0


In [30]:
invalid_log_values = {
    "num_25": 0,
    "num_50": 0,
    "num_75": 0,
    "num_985": 0,
    "num_100": 0,
    "num_unq": 0,
    "total_secs": 0
}

log_value_cols = list(invalid_log_values.keys())

for chunk in pd.read_csv(
    USER_LOGS,
    usecols=log_value_cols,
    chunksize=CHUNK_SIZE
):
    for col in log_value_cols:
        invalid_log_values[col] += (chunk[col] < 0).sum()

print(invalid_log_values)

{'num_25': np.int64(0), 'num_50': np.int64(0), 'num_75': np.int64(0), 'num_985': np.int64(0), 'num_100': np.int64(0), 'num_unq': np.int64(0), 'total_secs': np.int64(0)}


In [31]:
invalid_transaction_dates = 0

for chunk in pd.read_csv(
    TRANSACTIONS,
    usecols=["transaction_date"],
    chunksize=CHUNK_SIZE
):
    dates = chunk["transaction_date"].astype(str)

    invalid_transaction_dates += (
        ~dates.str.match(r"^\d{8}$")
    ).sum()

print("Invalid transaction date format:", invalid_transaction_dates)

Invalid transaction date format: 0


In [32]:
invalid_log_dates = 0

for chunk in pd.read_csv(
    USER_LOGS,
    usecols=["date"],
    chunksize=CHUNK_SIZE
):
    dates = chunk["date"].astype(str)

    invalid_log_dates += (
        ~dates.str.match(r"^\d{8}$")
    ).sum()

print("Invalid listening date format:", invalid_log_dates)

Invalid listening date format: 0
